# 06 — Budget convergence and the production posterior (claim C2)
160k bank (production conditions, protocol and features from `config.py`, or as decided in 03/05),
nested budgets 5k–160k × 5 seeds, evaluation on a fixed test set, **rule v2.1 unchanged** (`rules.budget_rule_v21`).
Writes `results/06_production.json`. ~2–4 h (training dominates).

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
bank_f = DATA / "06_bank_160k.npz"
if not bank_f.exists():
    da.save_npz(bank_f, da.make_windows(sim.prior_draws(160_000, 6100), 6100, protocol=cfg.PROTOCOL, feature_set=cfg.FEATURE_SET))
bank = da.load_npz(bank_f)
test_f = DATA / "06_test.npz"
if not test_f.exists():
    da.save_npz(test_f, da.test_set(6199, protocol=cfg.PROTOCOL, feature_set=cfg.FEATURE_SET))
test = da.load_npz(test_f)
OUT = R / "06"; OUT.mkdir(exist_ok=True)
f_rows = R / "06_metrics.csv"
rows = pd.read_csv(f_rows).to_dict("records") if f_rows.exists() else []
done = {(int(r["budget"]), str(r["member"])) for r in rows}
for N in cfg.BUDGETS:
    members = []
    for s in cfg.SEEDS:
        f = OUT / f"post_n{N}_s{s}.pkl"
        if f.exists():
            post = pickle.load(open(f, "rb"))
        else:
            idx = np.random.default_rng(36_000 + s).permutation(len(bank["theta"]))[:N]
            post, meta = inf.train_npe(bank["theta"][idx], bank["x"][idx], seed=s); pickle.dump(post, open(f, "wb"))
        members.append(post)
        if (N, str(s)) not in done:
            row, _ = inf.evaluate(post, test, seed=0); rows.append({"budget": N, "member": str(s), **row})
    if (N, "ensemble") not in done:
        row, _ = inf.evaluate(inf.EnsemblePosterior(members), test, seed=0); rows.append({"budget": N, "member": "ensemble", **row})
    pd.DataFrame(rows).to_csv(f_rows, index=False); print(N, "done", f"{(time.time() - T0) / 60:.1f} min", flush=True)

In [ ]:
met = pd.read_csv(f_rows)
dec = rules.budget_rule_v21(met)
dec.update(protocol=cfg.PROTOCOL, feature_set=cfg.FEATURE_SET, sensors=cfg.SENSORS, conditions=cfg.CONDITIONS,
           member_files=[str(OUT / f"post_n{dec['production_budget']}_s{m}.pkl") for m in dec["members"]])
da.save_json(R / "06_production.json", dec)
display(pd.DataFrame(dec["steps"])); print({k: v for k, v in dec.items() if k != "steps"})
if dec["flag"]:
    raise RuntimeError("STOP: " + dec["flag"])

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).